In [4]:
import pandas as pd
import asyncio
from utils_fbo_act import main_fbo

In [ ]:
if __name__ == "__main__":
    # asyncio.run(main_fbo())
    await main_fbo(100)

In [14]:
from utils_fbo_act import fbo_dict_docs, get_decoded_acts, load_api_tokens, create_acceptance_certificate_fbo, create_insert_table_db_sync, proccessing_data_acceptance_act


async def get_process_fbo_acts(days_back:int = 5):    # Получаем словарь документов ФБО за указанное количество дней назад
    days_back = 5
    try:
        dict_docs_fbo = await(fbo_dict_docs(days_back))
    except RuntimeError:
        dict_docs_fbo = await fbo_dict_docs(days_back)

    # Получаем декодированные акты ФБО асинхронно
    tasks = [asyncio.create_task(get_decoded_acts(account, doc_list, tokens=load_api_tokens())) for account, doc_list in dict_docs_fbo.items()]
    # Собираем все декодированные акты в список
    decoded_acts = await asyncio.gather(*tasks)
    return decoded_acts

processed_acts = await get_process_fbo_acts(2)

Успешно получены документы за период 2025-12-16 - 2025-12-16 по аккаунту Даниелян
Получено 2 документов, всего: 2, offset: 0
Последняя страница: всего получено 2 документов
Аккаунт Даниелян: получено 2 документов act-income
Успешно получены документы за период 2025-12-16 - 2025-12-16 по аккаунту Вектор
Получено 4 документов, всего: 4, offset: 0
Последняя страница: всего получено 4 документов
Аккаунт Вектор: получено 4 документов act-income
Успешно получены документы за период 2025-12-16 - 2025-12-16 по аккаунту Оганесян
Получено 4 документов, всего: 4, offset: 0
Последняя страница: всего получено 4 документов
Аккаунт Оганесян: получено 4 документов act-income
Успешно получены документы за период 2025-12-16 - 2025-12-16 по аккаунту Хачатрян
Получено 5 документов, всего: 5, offset: 0
Последняя страница: всего получено 5 документов
Аккаунт Хачатрян: получено 5 документов act-income
Успешно получены документы за период 2025-12-16 - 2025-12-16 по аккаунту Старт
Получено 4 документов, всего:

In [8]:
docs = []
for account, acts in decoded_acts[0].items():
    l = proccessing_data_acceptance_act(account, acts)
    for doc in l:
        docs.append(doc)
if docs:
    df = pd.concat(docs)
else:
    df = pd.DataFrame()

Файлы внутри вложенного ZIP:
  - mchd.zip
  - signature.sig
  - act-income-35319108.xlsx
Количество колонок в df: 9
Количество заголовков: 9
Данные по act-income-35319108.zip добавлены в список
Файлы внутри вложенного ZIP:
  - mchd.zip
  - signature.sig
  - act-income-35302823.xlsx
Количество колонок в df: 9
Количество заголовков: 9
Данные по act-income-35302823.zip добавлены в список
Файлы внутри вложенного ZIP:
  - act-income-35299225.xlsx
Количество колонок в df: 9
Количество заголовков: 9
Данные по act-income-35299225.zip добавлены в список
  - mchd.zip
  - signature.sig
Файлы внутри вложенного ZIP:
  - mchd.zip
  - signature.sig
  - act-income-35327616.xlsx
Количество колонок в df: 9
Количество заголовков: 9
Данные по act-income-35327616.zip добавлены в список
Файлы внутри вложенного ZIP:
  - mchd.zip
  - signature.sig
  - act-income-35283181.xlsx


C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\xml\etree\ElementTree.py:1299: RuntimeWarning: coroutine 'fbo_dict_docs' was never awaited
  self._parser.feed(data)


Количество колонок в df: 9
Количество заголовков: 9
Данные по act-income-35283181.zip добавлены в список
Файлы внутри вложенного ZIP:
  - mchd.zip
  - signature.sig
  - act-income-35279099.xlsx
Количество колонок в df: 9
Количество заголовков: 9
Данные по act-income-35279099.zip добавлены в список
Файлы внутри вложенного ZIP:
  - act-income-35277869.xlsx
Количество колонок в df: 9
Количество заголовков: 9
Данные по act-income-35277869.zip добавлены в список
  - mchd.zip
  - signature.sig
Файлы внутри вложенного ZIP:
  - mchd.zip
  - signature.sig
  - act-income-35277866.xlsx
Количество колонок в df: 9
Количество заголовков: 9
Данные по act-income-35277866.zip добавлены в список
Файлы внутри вложенного ZIP:
  - signature.sig
  - act-income-35277882.xlsx
Количество колонок в df: 9
Количество заголовков: 9
Данные по act-income-35277882.zip добавлены в список
  - mchd.zip
Файлы внутри вложенного ZIP:
  - mchd.zip
  - signature.sig
  - act-income-35279119.xlsx
Количество колонок в df: 9
Кол

In [12]:
# ---- test ----
docs = []
try:
    for account, acts in decoded_acts[0].items():
        l = proccessing_data_acceptance_act(account, acts)
        for doc in l:
            docs.append(doc)
    if docs:
        df = pd.concat(docs)
    else:
        df = pd.DataFrame()
except IndexError:
    print(decoded_acts)

if ' - ШК товара' in df.columns:
    # Из полученных данных формируем акты-приема передачи для ФБО
    fbo_acts_df = df[['№ п\п', 'Товар (наименование)', 'Ед. изм.', 'Фактически принято - баркод', ' - артикул продавца', ' - сорт, размер', ' - КИЗ', ' - ШК короба', ' - кол-во', 'Документ','Номер_документа', 'Дата', ' - ШК товара', 'account']]
else:
    df[' - ШК товара'] = 0
    # Из полученных данных формируем акты-приема передачи для ФБО
    fbo_acts_df = df[['№ п\п', 'Товар (наименование)', 'Ед. изм.', 'Фактически принято - баркод', ' - артикул продавца', ' - сорт, размер', ' - КИЗ', ' - ШК короба', ' - кол-во', 'Документ','Номер_документа', 'Дата', ' - ШК товара', 'account']]
# ---- test ----

# ВБ иногда путает акты ФБО и ФБС, поэтому фильтруем по ШК короба
fbo_acts_df = fbo_acts_df[fbo_acts_df[' - ШК короба'].notna()]

# Приводим названия колонок к читаемому виду
fbo_acts_df = fbo_acts_df.rename(columns={
    '№ п\\п': 'num',
    'Товар (наименование)': 'product_name',
    'Ед. изм.': 'unit',
    'Фактически принято - баркод': 'barcode',
    ' - артикул продавца': 'vendor_code',
    ' - сорт, размер': 'size',
    ' - КИЗ': 'kiz',
    ' - ШК короба': 'box_barcode',
    ' - кол-во': 'quantity',
    'Документ': 'document',
    'Номер_документа': 'document_number',
    'Дата': 'date',
    ' - ШК товара': 'shk_id',
    'account': 'account'
})

# Заменяем пустоты
fbo_acts_df['kiz'] = fbo_acts_df['kiz'].fillna('Нет КИЗов')

# Приводим колонку с датой к нужному формату
fbo_acts_df['date'] = fbo_acts_df['date'].str.replace('"','').str.replace(' ', '').str.replace('г.', '')
fbo_acts_df['date'] = pd.to_datetime(fbo_acts_df['date'], format='%d%m%Y', errors='coerce')

# Удаляем лишние символы из номера документа
fbo_acts_df['document_number'] = fbo_acts_df['document'].str.extract(r'(\d+)\.zip')[0]
    # Обработка FBO данных
if not fbo_acts_df.empty:
    fbo_acts_df = fbo_acts_df.where(pd.notnull(fbo_acts_df), None)  # NaN → None
    # Исправляем преобразование даты для FBO
    fbo_acts_df['date'] = pd.to_datetime(fbo_acts_df["date"], dayfirst=True, errors='coerce').dt.date
    fbo_acts_df['num'] = fbo_acts_df['num'].astype(int)
    # ВБ в какой-то момент убрал поле количество из формы акта ПП по ФБО. Поэтому меняем пустоты на единицу
    fbo_acts_df['quantity'] = fbo_acts_df['quantity'].fillna(1)
    fbo_acts_df['quantity'] = fbo_acts_df['quantity'].astype(int)
    # Поле shk_id появилось в акте позже. Поэтом предыдущие значения заполняем нулями
    fbo_acts_df['shk_id'] = fbo_acts_df['shk_id'].fillna(0)
    fbo_acts_df['shk_id'] = fbo_acts_df['shk_id'].astype(int)
    
print('Данные по ФБО получены')                  



# # Определяем типы колонок и ключевые колонки для записи в БД
# columns_type_fbo = {
#     'num': 'INTEGER',
#     'product_name': 'VARCHAR(255)',
#     'unit': 'VARCHAR(50)',
#     'barcode': 'VARCHAR(50)', 
#     'vendor_code': 'VARCHAR(50)',
#     'size': 'VARCHAR(50)',
#     'kiz': 'VARCHAR(255)',
#     'box_barcode': 'VARCHAR(50)',
#     'quantity': 'INTEGER',
#     'document': 'VARCHAR(255)',
#     'document_number': 'VARCHAR(50)',
#     'date': 'DATE',
#     'shk_id': 'BIGINT',
#     'account': 'VARCHAR(100)'
# }
# # Определяем ключевые колонки для обновления записей в БД
# key_cols_fbo = ('vendor_code', 'box_barcode', 'document_number','shk_id')
# # Указываем имя таблицы для записи
# table_name_fbo = 'acceptance_fbo_acts_new'
# # Записываем данные в таблицу БД
# create_insert_table_db_sync(fbo_acts_df, table_name_fbo, columns_type_fbo, key_cols_fbo)

Файлы внутри вложенного ZIP:
  - mchd.zip
  - signature.sig
  - act-income-35319108.xlsx
Количество колонок в df: 9
Количество заголовков: 9
Данные по act-income-35319108.zip добавлены в список
Файлы внутри вложенного ZIP:
  - mchd.zip
  - signature.sig
  - act-income-35302823.xlsx
Количество колонок в df: 9
Количество заголовков: 9
Данные по act-income-35302823.zip добавлены в список
Файлы внутри вложенного ZIP:
  - act-income-35299225.xlsx
Количество колонок в df: 9
Количество заголовков: 9
Данные по act-income-35299225.zip добавлены в список
  - mchd.zip
  - signature.sig
Файлы внутри вложенного ZIP:
  - mchd.zip
  - signature.sig
  - act-income-35327616.xlsx
Количество колонок в df: 9
Количество заголовков: 9
Данные по act-income-35327616.zip добавлены в список
Файлы внутри вложенного ZIP:
  - mchd.zip
  - signature.sig
  - act-income-35283181.xlsx


C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\xml\etree\ElementTree.py:1299: RuntimeWarning: coroutine 'fbo_dict_docs' was never awaited
  self._parser.feed(data)


Количество колонок в df: 9
Количество заголовков: 9
Данные по act-income-35283181.zip добавлены в список
Файлы внутри вложенного ZIP:
  - mchd.zip
  - signature.sig
  - act-income-35279099.xlsx
Количество колонок в df: 9
Количество заголовков: 9
Данные по act-income-35279099.zip добавлены в список
Файлы внутри вложенного ZIP:
  - act-income-35277869.xlsx
Количество колонок в df: 9
Количество заголовков: 9
Данные по act-income-35277869.zip добавлены в список
  - mchd.zip
  - signature.sig
Файлы внутри вложенного ZIP:
  - mchd.zip
  - signature.sig
  - act-income-35277866.xlsx
Количество колонок в df: 9
Количество заголовков: 9
Данные по act-income-35277866.zip добавлены в список
Файлы внутри вложенного ZIP:
  - signature.sig
  - act-income-35277882.xlsx
Количество колонок в df: 9
Количество заголовков: 9
Данные по act-income-35277882.zip добавлены в список
  - mchd.zip
Файлы внутри вложенного ZIP:
  - mchd.zip
  - signature.sig
  - act-income-35279119.xlsx
Количество колонок в df: 9
Кол

C:\Users\123\AppData\Local\Temp\ipykernel_10492\2674121551.py:61: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  fbo_acts_df['quantity'] = fbo_acts_df['quantity'].fillna(1)


In [13]:
fbo_acts_df

,num,product_name,unit,barcode,vendor_code,size,kiz,box_barcode,quantity,document,document_number,date,shk_id,account
0,1,Казаны,шт.,2043229313412,wild1522,0,Нет КИЗов,WB_1500131379,40,act-income-35319108.zip,35319108,2025-12-16,0,Вектор
1,2,Казаны,шт.,2042957454749,wild1514,0,Нет КИЗов,WB_1500131383,36,act-income-35319108.zip,35319108,2025-12-16,0,Вектор
2,3,Казаны,шт.,2042957454770,wild1515,0,Нет КИЗов,WB_1500131383,23,act-income-35319108.zip,35319108,2025-12-16,0,Вектор
3,4,Казаны,шт.,2042957454817,wild1513,0,Нет КИЗов,WB_1500131380,36,act-income-35319108.zip,35319108,2025-12-16,0,Вектор
4,5,Казаны,шт.,2042957454770,wild1515,0,Нет КИЗов,WB_1500131382,45,act-income-35319108.zip,35319108,2025-12-16,0,Вектор
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54,55,Наборы кухонных принадлежностей,шт.,2037636331121,wild355,0,Нет КИЗов,WB_1496988731,8,act-income-35188397.zip,35188397,2025-12-13,0,Вектор
55,56,Наборы кухонных принадлежностей,шт.,2037636331121,wild355,0,Нет КИЗов,WB_1496988711,8,act-income-35188397.zip,35188397,2025-12-13,0,Вектор
56,57,Наборы кухонных принадлежностей,шт.,2037636331121,wild355,0,Нет КИЗов,WB_1496988715,8,act-income-35188397.zip,35188397,2025-12-13,0,Вектор
57,58,Наборы кухонных принадлежностей,шт.,2037636331121,wild355,0,Нет КИЗов,WB_1496988717,8,act-income-35188397.zip,35188397,2025-12-13,0,Вектор
